# 🚀 GARRO: Research-Grade DRL Routing Orchestrator for Data Centers
## ⚡ Kaggle Cloud Training Pipeline: Dual Tesla T4 GPUs (31.2 GB VRAM) | 4 CPU Cores | 30 GB RAM

This notebook trains **GARRO** (Graph Attention Routing with Reinforcement Learning) on a **Fat-Tree ($k=4$) Data Center Network (DCN)** fabric under production traffic conditions.

### 🌟 Key Innovations
1. **Path-Centric Attention (RouteNet-Fermi Paradigm)**: Replaces global MLP pooling with explicit constituent link and bottleneck aggregation ($h_{\text{bottleneck}} = \max_{e \in P} h_e$) to eliminate ECMP collisions.
2. **Realistic DCN Traffic Dynamics**: Diurnal dual-wave time-of-day modulation, Pareto elephant flows ($\alpha=1.15$), incast microbursts, bisection stride stress, and AllReduce collective exchange.
3. **Multi-GPU Parallelism (DDP)**: Distributed Data Parallel training across Dual Tesla T4 GPUs utilizing NCCL ring-allreduce.
4. **Vectorized Multi-Environment Rollout**: $N=32$ parallel environments ($16$ envs/GPU) to saturate Turing Tensor Cores and bypass the 4-CPU bottleneck.
5. **3-Stage Curriculum Learning**: Warmup Exploration $\to$ Heavy Bimodal Stress $\to$ Chaos & Incast Collective Bursts.
6. **Persistent Offline Cloud Execution**: Run in the background via Kaggle's *Save & Run All (Commit)* so you can close your browser or go offline without losing progress.

## 1. Import or Synchronize GitHub Repository
Clones or pulls the latest GARRO codebase into the Kaggle environment.

In [ ]:
import os
import sys

# If running directly inside the cloned repo, use current dir; otherwise clone from GitHub
if not os.path.exists("config.yaml"):
    print("[Repo] Cloning latest GARRO repository from GitHub...")
    !git clone https://github.com/DanielAgbeni/GARRO.git /kaggle/working/GARRO
    %cd /kaggle/working/GARRO
else:
    print("[Repo] Found existing repository in current directory.")
    !git pull origin main || true

print(f"Active Working Directory: {os.getcwd()}")
print("Repo files:", os.listdir(".")[:8])

## 2. Hardware & Accelerator Verification
Confirm that both Tesla T4 GPUs, 4 CPU cores, and 30 GB RAM are accessible.

In [ ]:
!nvidia-smi

import multiprocessing
import psutil
import torch

n_gpus = torch.cuda.device_count()
n_cpus = multiprocessing.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024 ** 3)

print(f"\n{'='*60}")
print(f"  Kaggle Hardware Verification Report")
print(f"{'='*60}")
print(f"  GPUs Detected     : {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"    GPU {i}: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")
print(f"  CPU Cores         : {n_cpus}")
print(f"  System RAM        : {ram_gb:.2f} GB")
print(f"  CUDA Available    : {torch.cuda.is_available()}")
if n_gpus >= 2:
    print("  STATUS            : READY FOR DUAL T4 DDP ACCELERATION ✓")
else:
    print("  STATUS            : WARNING: Set Accelerator to 'GPU T4 x2' in Kaggle Settings")
print(f"{'='*60}")

## 3. Install Dependencies & Build Environment
Installs PyTorch Geometric, NetworkX, SciPy, Matplotlib, and other required libraries.

In [ ]:
!pip install --quiet torch-geometric networkx scipy pyyaml matplotlib tqdm psutil tabulate
print("All core dependencies verified and installed successfully! ✓")

## 4. 🔍 Pre-Flight Smoke Test (< 15 Seconds)
**Run this before the full training.** Tests all 7 subsystems to catch any irregularity immediately:
1. Hardware & GPU Acceleration
2. Fat-Tree Fabric & Topology
3. DCN Traffic Engine & Diurnal Wave
4. Path-Centric Actor-Critic & Bottleneck Pooling
5. Vectorized Environment Step & Reward Signals
6. PPO Loss Calculation & Backward Gradient Step
7. Checkpoint Serialization & Reload Integrity

In [ ]:
!python smoke_test_kaggle.py

## 5. Configure Persistent Storage & Checkpoint Directory
Ensures checkpoints are saved to `/kaggle/working/checkpoints/` so they persist even if you disconnect or close your browser.

In [ ]:
import os

ckpt_dir = "/kaggle/working/checkpoints" if os.path.exists("/kaggle/working") else "checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
print(f"Persistent Checkpoints Directory configured at: {os.path.abspath(ckpt_dir)}")

# Check if previous checkpoint exists for automated resume
latest_ckpt = os.path.join(ckpt_dir, "garro_fat_tree_latest.pt")
if os.path.exists(latest_ckpt):
    print(f"[Resume] Found existing checkpoint: {latest_ckpt}. Training will automatically resume!")
else:
    print("[Fresh] No previous checkpoint found. Starting fresh curriculum run.")

## 6. Launch Multi-GPU DDP 50,000-Episode Curriculum Training
Executes `torchrun --nproc_per_node=2` to train GARRO across both Tesla T4 GPUs.

> **💡 How to train offline:**
> Click **"Save Version"** in the top-right corner of Kaggle, choose **"Save & Run All (Commit)"**, and submit. You can turn off your computer or close your browser—Kaggle will run the training in the background for up to 9 hours and save all checkpoints automatically!

In [ ]:
!torchrun --nproc_per_node=2 train_kaggle.py \
    --topology fat_tree \
    --episodes 50000 \
    --num-envs 16 \
    --checkpoint-every 2500 \
    --checkpoint-dir checkpoints

## 7. Live Telemetry & Training Metrics Dashboard
Displays the moving-average reward curve and PPO actor-critic loss progression.

In [ ]:
from IPython.display import Image, display
import os

curve_img = "checkpoints/training_curve_fat_tree.png"
if os.path.exists(curve_img):
    display(Image(curve_img))
else:
    print("Training curve image not yet generated. Run training first.")

## 8. Multi-Checkpoint Benchmark Evaluation & Automatic Model Selection
Benchmarks ALL saved curriculum checkpoints in `/kaggle/working/checkpoints/` against OSPF, strict equal-cost ECMP, and dynamic Hedera to automatically detect and rank the 🏆 optimal model (since the final epoch might not always be the best policy due to late-stage exploration).

In [ ]:
import os

ckpt_dir = "/kaggle/working/checkpoints" if os.path.exists("/kaggle/working/checkpoints") else "checkpoints"

if os.path.exists(ckpt_dir) and any(f.endswith('.pt') for f in os.listdir(ckpt_dir)):
    print(f"[Eval] Benchmarking ALL checkpoints found in '{ckpt_dir}'...")
    print("[Eval] Evaluates curriculum progression, compares against baselines, and crowns the 🏆 BEST model!")
    !python evaluate.py \
        --topology fat_tree \
        --checkpoint {ckpt_dir} \
        --episodes 500 \
        --compare-all
else:
    print(f"No .pt checkpoints found in '{ckpt_dir}'. Complete training step first.")

## 9. Package Checkpoints & Downloadable Archive
Compresses all checkpoints, logs, and benchmark plots into `garro_kaggle_trained_artifacts.tar.gz` for 1-click download from Kaggle's output pane.

In [ ]:
!tar -czvf /kaggle/working/garro_kaggle_trained_artifacts.tar.gz checkpoints/ *.png 2>/dev/null || tar -czvf garro_kaggle_trained_artifacts.tar.gz checkpoints/ *.png
print("\n✓ Artifacts packaged successfully! Available in the right-hand Output pane for download.")